##### Introduction
This notebook demonstrates how to automate Fabric data agent functionalities such as creating a data agent; adding a datasource (e.g. Lakehouse) or adding instructions to a data agent programmatically using our library. More information on data agent can be found [here](https://learn.microsoft.com/en-us/fabric/data-science/concept-ai-skill).

In [ ]:
%pip install fabric-data-agent-sdk

In [ ]:
from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

## First, let's create a data agent

In [ ]:
# create DataAgent
data_agent_name = "Quickstart data agent"
workspace_id = "835d3ceb-5ddd-4b25-a1aa-bd8448094cac"

data_agent = create_data_agent(
    data_agent_name=data_agent_name,
    workspace_id=workspace_id,
)

# data_agent = create_data_agent(data_agent_name)

### We can check the configuration of a data agent as shown below

In [ ]:
# by default the instructions and description for the data agent will be empty, we will update them later in the notebook
data_agent.get_settings()

### We can also initialize a client for an existing data agent

In [ ]:
# data_agent = FabricDataAgentManagement(data_agent_name)

### Update data agent by adding AI instructions

In [ ]:
data_agent.update_settings(
    ai_instructions="You are a helpful assistant, help users with their questions",
)
data_agent.get_settings()

### Add Data Source
We will now add a datasource to our data agent. In this sample we will add a Lakehouse as a datasource, however we can also add Semantic Model or Kusto

In [ ]:
# add a lakehouse
lakehouse_name = "Gaming_LH"

# datasource type could be: lakehouse, kqldatabase, warehouse or semanticmodel
data_agent.add_staging_datasource(lakehouse_name, type="lakehouse")

### List Data Source

In [ ]:
# we can check which datasources are added to the data agent
data_agent.list_datasources()

### We can publish the data agent

In [ ]:
data_agent.publish_staging(description="Initial publish")

Now we will work with the datasource we just added

In [ ]:
datasource = data_agent.list_datasources()[0]

We can also add few-shot examples

In [ ]:
datasource.add_fewshot(
    question="Show the top 10 players by total purchase amount.",
    query="""
    SELECT TOP 10
        pl.PlayerName,
        SUM(p.Amount) AS TotalSpent
    FROM purchases p
    JOIN players pl
        ON p.PlayerID = pl.PlayerID
    GROUP BY pl.PlayerName
    ORDER BY TotalSpent DESC;
    """
)

In [ ]:
fewshots = [
    {
        "question": "What is the total revenue?",
        "query": """
            SELECT SUM(Amount) AS TotalRevenue
            FROM purchases;
        """
    },
    {
        "question": "List all players from India.",
        "query": """
            SELECT PlayerName, Country
            FROM players
            WHERE Country = 'India';
        """
    },
    {
        "question": "Show total purchase amount by country.",
        "query": """
            SELECT
                pl.Country,
                SUM(p.Amount) AS TotalRevenue
            FROM purchases p
            JOIN players pl
                ON p.PlayerID = pl.PlayerID
            GROUP BY pl.Country
            ORDER BY TotalRevenue DESC;
        """
    },
    {
        "question": "Show the top 5 purchased items.",
        "query": """
            SELECT TOP 5
                Item,
                COUNT(*) AS PurchaseCount
            FROM purchases
            GROUP BY Item
            ORDER BY PurchaseCount DESC;
        """
    },
    {
        "question": "How many players signed up each month?",
        "query": """
            SELECT
                YEAR(SignupDate) AS Year,
                MONTH(SignupDate) AS Month,
                COUNT(*) AS PlayerCount
            FROM players
            GROUP BY
                YEAR(SignupDate),
                MONTH(SignupDate)
            ORDER BY
                Year,
                Month;
        """
    }
]

for fs in fewshots:
    datasource.add_fewshot(
        question=fs["question"],
        query=fs["query"]
    )

In [ ]:
datasource.get_fewshots()

In [ ]:
# Publish the changes
data_agent.publish_staging()

We can delete few-shots using their ids

In [ ]:
# make sure the replace the id of the few-shot example with the id that is assigned to your few-shot example
datasource.delete_fewshot("5a8fa7d9-34f8-4d8d-ae2b-45554281f154")

In [ ]:
datasource.get_fewshots()

Finally, we can delete the data agent

In [ ]:
delete_data_agent(data_agent_name)